# 🧠 HybridID — S5: Hyperparameter Tuning (KerasTuner)
Bu notebook, ResNet50 modelinin sınıflandırıcı katmanındaki hiperparametreleri optimize etmek için kullanılır.

In [ ]:
!pip install keras-tuner

from google.colab import drive
drive.mount('/content/drive')

# Dataset'i zip'ten çıkar (Eğer çıkmamışsa)
import os
ZIP_PATH = '/content/drive/MyDrive/processed.zip'
EXTRACT_DIR = '/content/processed'
if not os.path.exists(EXTRACT_DIR):
    !unzip -q "{ZIP_PATH}" -d /content/
    print('Dataset hazır!')

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import keras_tuner as kt

CONFIG = {
    'img_size': (224, 224),
    'batch_size': 64,
    'dataset_dir': EXTRACT_DIR,
    'model_dir': '/content/drive/MyDrive/HybridID_models'
}

# Data Generators
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True, rotation_range=20,
    width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.15, brightness_range=[0.85, 1.15], shear_range=5,
)
eval_datagen = keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    os.path.join(CONFIG['dataset_dir'], 'train'),
    target_size=CONFIG['img_size'], batch_size=CONFIG['batch_size'],
    class_mode='binary', shuffle=True, seed=42
)
val_gen = eval_datagen.flow_from_directory(
    os.path.join(CONFIG['dataset_dir'], 'val'),
    target_size=CONFIG['img_size'], batch_size=CONFIG['batch_size'],
    class_mode='binary', shuffle=False
)

In [ ]:
def build_tunable_model(hp):
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(*CONFIG['img_size'], 3))
    base.trainable = False

    hp_dense_units = hp.Choice('dense_units', values=[128, 256, 512])
    hp_dropout_rate = hp.Choice('dropout_rate', values=[0.3, 0.5, 0.7])
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])
    hp_l2_lambda = hp.Choice('l2_lambda', values=[1e-4, 1e-5])

    inputs = keras.Input(shape=(*CONFIG['img_size'], 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(hp_dense_units, activation="relu", kernel_regularizer=regularizers.l2(hp_l2_lambda))(x)
    x = layers.Dropout(hp_dropout_rate)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss="binary_crossentropy", metrics=["accuracy"]
    )
    return model

tuner = kt.RandomSearch(
    build_tunable_model, objective='val_accuracy', max_trials=8, executions_per_trial=1,
    directory=os.path.join(CONFIG['model_dir'], "tuning"), project_name='resnet50_head_tune'
)

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\n[Tuning] Arama başlatılıyor...")
tuner.search(train_gen, validation_data=val_gen, epochs=8, callbacks=[early_stop])

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\n🌟 EN İYİ HİPERPARAMETRELER:")
print(f"- Dense Units: {best_hps.get('dense_units')}")
print(f"- Dropout Rate: {best_hps.get('dropout_rate')}")
print(f"- Learning Rate: {best_hps.get('learning_rate')}")
print(f"- L2 Lambda: {best_hps.get('l2_lambda')}")